# TPC Ablation Study - Google Colab

**Steps:**
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run cells 1-3 to setup and install dependencies
3. Run cell 4 to mount Google Drive (for saving results)
4. Run cell 5 to download MIMIC-IV data (~10-15 min, 9.92 GB)
5. Run cell 6 to configure paths
6. Run cell 7 to start training (2-4 hours)
7. Results save to MyDrive/tpc_ablation_results/

In [ ]:
# Clone repository
!git clone https://github.com/tarakjc2c/PyHealth.git
%cd PyHealth
!git checkout pr-1028

In [ ]:
# Install dependencies
!pip install -e . -q
!pip install litdata polars pandas dask mne rdkit peft transformers ogb -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Download MIMIC-IV data from shared Google Drive
!pip install gdown -q
import os

MIMIC_ROOT = '/content/mimic-iv'
GDRIVE_LINK = 'https://drive.google.com/drive/folders/15vyfKQ6H0g7DVEbI8vAMNhr4fi2lznVn?usp=sharing'

if not os.path.exists(MIMIC_ROOT):
    print('Downloading MIMIC-IV data (9.92 GB, ~10-15 minutes)...')
    !gdown --folder {GDRIVE_LINK} -O /content --quiet
    print('✓ Download complete')
else:
    print('✓ MIMIC-IV data already downloaded')

# Verify files
hosp_files = len([f for f in os.listdir(f'{MIMIC_ROOT}/hosp') if f.endswith('.csv.gz')])
icu_files = len([f for f in os.listdir(f'{MIMIC_ROOT}/icu') if f.endswith('.csv.gz')])
print(f'hosp/: {hosp_files} files')
print(f'icu/: {icu_files} files')

In [ ]:
# Fix paths for Colab
script = 'examples/length_of_stay/length_of_stay_mimic4_tpc.py'

with open(script, 'r') as f:
    lines = f.readlines()

# Replace path definitions
new_lines = []
for line in lines:
    if 'MIMIC_ROOT = r"C:' in line:
        new_lines.append('MIMIC_ROOT = "/content/mimic-iv"\n')
    elif 'CACHE_PATH = r"C:' in line:
        new_lines.append('CACHE_PATH = "/content/tpc_cache"\n')
    elif 'OUTPUT_DIR = "tpc_ablation_results"' in line:
        new_lines.append('OUTPUT_DIR = "/content/drive/MyDrive/tpc_ablation_results"\n')
    else:
        new_lines.append(line)

with open(script, 'w') as f:
    f.writelines(new_lines)

print('✓ Paths configured for Colab')

In [ ]:
# Run ablation study (2-4 hours)
!python examples/length_of_stay/length_of_stay_mimic4_tpc.py

## Download Results

Results are in MyDrive/tpc_ablation_results/
Download them with the cell below:

In [ ]:
from google.colab import files
import os

result_dir = '/content/drive/MyDrive/tpc_ablation_results'
for filename in ['ablation_results.json', 'mc_dropout_results.json']:
    filepath = os.path.join(result_dir, filename)
    if os.path.exists(filepath):
        print(f'Downloading: {filename}')
        files.download(filepath)